# Load a model

Pick one of the two ways below. Both end with the same `ata.run(...)` call.

| | Use it when |
|---|---|
| Local | You download the weights and run `llama-server` on this computer |
| Remote | One `llama-server` is already running with the merged GGUF |

With `models`, the marker score and the name model each return a result. The check rule keeps that name, replaces it, or drops it. The marker score can still name the cluster when the check drops the proposal. The title is a dictionary name. The name model copies a cell-type name from the marker dictionary. The check is not a second model. A cluster can stay unnamed. Without `models`, only the marker score is collected.


## Local

The weights are on the Hugging Face Hub: [lieberlee/annotune-qwen3.5-4b-gguf](https://huggingface.co/lieberlee/annotune-qwen3.5-4b-gguf). The current file is one merged Q6_K GGUF, `Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf`. Pass that file to `-m`.

The first `run` starts `http://127.0.0.1:8080` with that file. A later `run` reuses the server when the same file is already loaded.

### 1. Install the launcher

Do this once. Skip it when `llama-server` is already installed. The build is [llama.cpp](https://github.com/ggml-org/llama.cpp) **b12180**.

| System | Support |
|---|---|
| Linux with an NVIDIA GPU | Supported. Steps below. |
| macOS (Apple silicon or Intel) | Supported. Use the macOS steps below. The Metal build uses the GPU automatically. |
| Windows | This package does not start `llama-server`. Use [AnnoTuneLoader](https://github.com/leelieber2025/AnnoTuneLoader). |

On Windows, download [AnnoTuneLoader.exe](https://github.com/leelieber2025/AnnoTuneLoader/raw/main/AnnoTuneLoader.exe) and double-click it. Choose `Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf`. The server listens at `http://127.0.0.1:8080`. Leave the window open while you annotate. Another computer uses the Remote section below.

**Linux, CUDA 12.8.** Unpack the server and the CUDA runtime next to each other.

```bash
mkdir -p "$HOME/llama"
cd "$HOME/llama"
curl -L -o llama-cuda.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/llama-b12180-bin-ubuntu-cuda-12.8-x64.tar.gz
curl -L -o cudart.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/cudart-llama-b12180-bin-ubuntu-cuda-12.8-x64.tar.gz
tar -xzf llama-cuda.tar.gz
tar -xzf cudart.tar.gz
export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
"$LLAMA_SERVER" --version
```

The version line contains `build 12180`.

**macOS.** Pick `arm64` for Apple silicon and `x64` for Intel. There is no CUDA runtime to unpack.

```bash
mkdir -p "$HOME/llama"
cd "$HOME/llama"
curl -L -o llama-macos.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/llama-b12180-bin-macos-arm64.tar.gz
tar -xzf llama-macos.tar.gz
export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
xattr -dr com.apple.quarantine "$HOME/llama/llama-b12180"
"$LLAMA_SERVER" --version
```

The `xattr` line clears the download quarantine flag so macOS can run the binary. The merged file needs about 4 GB of memory at context 2048.


### 2. Download the weights

The file is about 3.3 GB. `hf download` resumes if the transfer stops.

`huggingface_hub` comes from `pip install huggingface_hub`.


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

model = Path(hf_hub_download(
    "lieberlee/annotune-qwen3.5-4b-gguf",
    "Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf",
    local_dir=Path.home() / "annotune-models",
))
print(model)


The file is `Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf`. Check it against the published checksum:


In [ ]:
import hashlib

expected = "4a90320205c32472b94172837499883322d360d80388998ca476a543fe0f43a5"
h = hashlib.sha256()
with open(model, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        h.update(block)
print("ok" if h.hexdigest() == expected else "BAD", model.name)


### 3. Load

Set `LLAMA_SERVER` to the `llama-server` file from step 1, then run the cell. The tokenizer is installed with the package. `loaded:` is `http://127.0.0.1:8080` when a server is already running, and `None` when it is not. This cell does not start a server.

In [ ]:
import os

import annotuneagent as ata

os.environ["LLAMA_SERVER"] = "/path/to/llama-server"

print("loaded:", ata.local_server())
model = "/path/to/Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf"
models = ata.ModelChain(
    name=ata.TaskEndpoint(base=model, maxlen=2048, server="http://127.0.0.1:8080"),
    ml=ata.TaskEndpoint(),
)
print("server:", models.name.server)

The server's state file and log live in a private folder: `$XDG_RUNTIME_DIR/annotune`, or `annotune-<user>` under the system temp folder. The folder is created with mode 0700. If it exists but belongs to another user, is a symlink, or is open to group or others, `run` stops with an error instead of reusing it. Delete or fix that folder and run again.

If a server on that address was started with different weights, `run` stops with `server already runs a different GGUF or context`. Stop that `llama-server` process and run again.

### 4. Annotate

`adata` comes from `ata.prepare`. Run this in the same notebook, after the cell above.

```python
report = ata.run(adata, "leiden", species="human", models=models)
print(ata.local_server())
```

The first call starts the server and leaves it running. The next call prints:

```text
server already loaded at http://127.0.0.1:8080; not loading again
```

`run` changes `adata` in place: it adds `annotune_*` columns and removes any stale `annotune_sub` column first. With `models` set, the marker score and the name model each return a result for every cluster, including clusters the marker score already named. The check rule writes the title. The generated `annotation_code` in the report expects the annotated object (it asserts that the cluster column exists), not the raw input.


## Remote

Use this when `llama-server` is already running with the merged file, including [AnnoTuneLoader](https://github.com/leelieber2025/AnnoTuneLoader) on another machine. Point the name task at that server. Each request calls `/completion`. If the server was started with `--api-key`, set `ANNOTUNE_API_KEY` to the same value. The package sends it as a bearer token. No local GGUF is required. Name prompts must fit in 2048 tokens.


In [ ]:
import annotuneagent as ata

server = "http://SERVER:8080"
models = ata.ModelChain(
    name=ata.TaskEndpoint(url=server, maxlen=2048),
    ml=ata.TaskEndpoint(),
)


Then annotate the same way as the local section:

```python
report = ata.run(adata, "leiden", species="human", models=models)
```

## Command line

Download the merged file, then pass it as `--base`. `--server` is the address. An existing local server is reused when that same file is already loaded.

```bash
hf download lieberlee/annotune-qwen3.5-4b-gguf \
  Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf \
  --local-dir "$HOME/annotune-models"

export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
export ANNOTUNE_URL=http://127.0.0.1:8080
annotuneagent --input counts.h5ad --prepare --species human \
  --hypotheses hypotheses.json \
  --server "$ANNOTUNE_URL" \
  --base "$HOME/annotune-models/Qwen3.5-4B-annotune-v9k-name.Q6_K.gguf" \
  --output result.json --output-h5ad labeled.h5ad
```

`--output-h5ad` is required when `--base` is set. The output folders are created and checked for write access before any model work starts, so a bad `--output` path fails at once. The report JSON is written before the h5ad.